### STEP 1: Importing relevant libraries

In [1]:
pip install optuna lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import optuna
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier)
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## STEP 2: Load the data

In [3]:
cleaned_df = pd.read_csv("../data/processed/cleaned_data.csv")
cleaned_df.head()

,amount_tsh,gps_height,population,age,month_recorded,permit,waterpoint_type_group,source_class,quantity,quality_group,payment_type,management_group,extraction_type_class,region,basin,status_group
0,6000.0,1390,109,12.0,3,False,communal_standpipe,groundwater,enough,good,annually,user-group,gravity,Iringa,Lake_Nyasa,functional
1,0.0,1399,280,3.0,3,True,communal_standpipe,surface,insufficient,good,never_pay,user-group,gravity,Mara,Lake_Victoria,functional
2,25.0,686,250,4.0,2,True,communal_standpipe,surface,enough,good,per_bucket,user-group,gravity,Manyara,Pangani,functional
3,0.0,263,58,27.0,1,True,communal_standpipe,groundwater,dry,good,never_pay,user-group,submersible,Mtwara,Ruvuma/Southern_Coast,non_functional
4,20.0,0,1,2.0,3,True,communal_standpipe,unknown,enough,salty,per_bucket,user-group,submersible,Tanga,Pangani,functional


## STEP 3: Split the features from the target variable, then into Training and Testing Sets

### Split the features from the target variable

In [4]:
X = cleaned_df.drop(columns=['status_group'])
y = cleaned_df['status_group']

print(display(X.head()))
print(display(y.head()))

,amount_tsh,gps_height,population,age,month_recorded,permit,waterpoint_type_group,source_class,quantity,quality_group,payment_type,management_group,extraction_type_class,region,basin
0,6000.0,1390,109,12.0,3,False,communal_standpipe,groundwater,enough,good,annually,user-group,gravity,Iringa,Lake_Nyasa
1,0.0,1399,280,3.0,3,True,communal_standpipe,surface,insufficient,good,never_pay,user-group,gravity,Mara,Lake_Victoria
2,25.0,686,250,4.0,2,True,communal_standpipe,surface,enough,good,per_bucket,user-group,gravity,Manyara,Pangani
3,0.0,263,58,27.0,1,True,communal_standpipe,groundwater,dry,good,never_pay,user-group,submersible,Mtwara,Ruvuma/Southern_Coast
4,20.0,0,1,2.0,3,True,communal_standpipe,unknown,enough,salty,per_bucket,user-group,submersible,Tanga,Pangani


None


0        functional
1        functional
2        functional
3    non_functional
4        functional
Name: status_group, dtype: object

None


In [5]:
### Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
 random_state=42, stratify=y)

Why stratify=y is important for imbalanced datasets

When splitting the data into ttrain & test sets, the goal is to ensure 
that both sets are representing the original dataset as closely as possible.

Without propper splitting, the minority classes may become underrepresented 
or even may entirely dissappear from either the training or the testing set.

### STEP 4: Encode the target variable (if not already encoded)

In [6]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

print("Encoded class labels and their corresponding interger values:",
dict(zip(le.classes_, le.transform(le.classes_))))

Encoded class labels and their corresponding interger values: {'functional': np.int64(0), 'functional_needs_repair': np.int64(1), 'non_functional': np.int64(2)}


### STEP 5: Build the preprocessing pipelines

In [7]:
# Separate  features into numerical and categorical
num_cols = X_train.select_dtypes(include=[np.number]).columns
cat_cols = X_train.select_dtypes(include=['object', 'bool', 'category']).columns

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: Index(['amount_tsh', 'gps_height', 'population', 'age', 'month_recorded'], dtype='object')
Categorical columns: Index(['permit', 'waterpoint_type_group', 'source_class', 'quantity',
       'quality_group', 'payment_type', 'management_group',
       'extraction_type_class', 'region', 'basin'],
      dtype='object')


### 1. Numerical Pipeline
The main transformation on the numerical pipeline is scaling, imputation of null.

_Choosing the best/right scaler_
Feature scaling is the process of transforming numerical features so they exist within a comparable range.

Scaling is important because many machine learning algorithm are sensitive to the magnitude of features.

Example:
|Feature|Values|
|-------|------|
|Age|18-80|
|Salary|20,000-2,000,000|

Without scaling, salary dominates calculations, optimization becomes unstable and some models perform poorly.

_Main types of scaler in sklearn_
|Scaler|Main Purpose|
|------|------------|
|StandardScaler|Standardization|
|MinMaxScaler|Normalization|
|RobustScaler|Outlier-resistant scaling|
|MaxAbsScaler|Sparse data|
|QuantileTransformer|Non-normal distributions|
|PowerTransformer|Reduce Skewness|


In [8]:
num_pipeline = Pipeline([('scaler', MinMaxScaler())])

# You can impute missing values using the pipeline as well, for example:
# num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', MinMaxScaler())])

### 2. Categorical Pipeline
The main transformations under the categorical pipeline is imputation of nulls and encoding
**Encoding** converts categorical data into numerical form so machine learning models can process it.
1. **Label Encoding**
- Assigns a unique interger to each category.
- Best fir for target variables and ordinal categories

_Example:_
|Color|Before Encoding|After Encoding|
|-----|---------------|--------------|
|Red|Red|0|
|Green|Green|1|
|Blue|Blue|2|
|Red|Red|0|

Note: It is not ideal for nominal features because models may assume numerical order exists.
Red > Green > Blue

2. **Onehot Encoding**
Creates a binary column for each category
Best for:
nominal categorical features.
non-ordered categories.

Before encoding:
|color|
|-----|
|Red|
|Green|
|Blue|

After encoding:
|Color_Red|Color_Green|Color_blue|
|---------|-----------|----------|
|1|0|0|
|0|1|0|
|0|0|1|

Advantages; prevents false ordering, works well in most ML models.
Disadvantages; increases dimensionality(number of columns), can create sparse matrices

3. **Ordinal Encoding**  
Assigns ordered integer based on category ranking.
_Example_  
Primary > High School > University

Before Encoding:
|Education(Before encoding)|After Encoding|
|--------------------------|--------------|
|Primary                   |             0|
|Secondary                 |             1|
|Bachelors                 |2             |
|Masters                   |3             |     

Note: Use only when true order exists


In [9]:
cat_pipeline = Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore'))])


# handle_unknown='ignore' ensures that if there are categories in the test 
# set that were not seen during training, the encoder will not raise an error and will instead create a new column for those unseen categories.

#### Combine the numerical and categorical pipeline

In [10]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

#### Create the model(if you have [selected the best baseline model] and after hyper parameter tuning the best model)
Note: use the best parameters to tune the models

In [13]:
full_pipeline = Pipeline([
   ('preprocessor', preprocessor),
   ('classifier', RandomForestClassifier(n_estimators=300, max_depth=20, min_samples_split=2, random_state=42))
 ])

 # Train the model
full_pipeline.fit(X_train, y_train_encoded)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [16]:
y_pred = full_pipeline.predict(X_test)
# Classification report
print("Classification Report:")
print(classification_report(y_test_encoded, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.91      0.85      4144
           1       0.56      0.27      0.36       478
           2       0.84      0.76      0.80      2731

    accuracy                           0.81      7353
   macro avg       0.73      0.64      0.67      7353
weighted avg       0.80      0.81      0.80      7353



## Save Model

In [17]:
import joblib
# Label encoder
joblib.dump(le, '../models/label_encoder.joblib')

# save the model
joblib.dump(full_pipeline, "../models/best_model.joblib")

['../models/best_model.joblib']

In [ ]:
# Checking the size of the joblib model
import os
filepath = "../models/best_model.joblib"
file_size = os.path.getsize(filepath)
print(f"Random Forest model Size: {file_size} bytes")

## Randomforest model is huge we wont use it

## STEP 6: (Baseline Model Selection) Run all possible models with the full pipeline and evaluate them using the F!-score metric.

In [12]:
models = {
    'LogisticRegression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000))]),

    'Decision Tree': Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeClassifier(class_weight='balanced'))]),

    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(class_weight='balanced', random_state=42))]),

    'Extra Trees': Pipeline([
        ('preprocessor', preprocessor),
        ('model', ExtraTreesClassifier(class_weight='balanced', random_state=42))]),

    'Gradient Boosting': Pipeline([
        ('preprocessor', preprocessor),
        ('model', GradientBoostingClassifier(random_state=42))]),

    'SVC': Pipeline([
        ('preprocessor', preprocessor),
        ('model', SVC(class_weight='balanced', random_state=42))]),

    'KNN': Pipeline([
        ('preprocessor', preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=5))]),

    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('model', xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42))]),

    'LightGBM': Pipeline([
        ('preprocessor', preprocessor),
        ('model', lgb.LGBMClassifier(random_state=42))])
}

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_encoded)

f1_results = []
for name, model in models.items():
    print(f"Training {name}...")
    if name in ['XGBoost', 'LightGBM', 'Gradient Boosting']:
        model.fit(X_train, y_train_encoded, model__sample_weight=sample_weights)
        
    else:
        model.fit(X_train, y_train_encoded)  
        
    y_pred = model.predict(X_test)
    report = classification_report(y_test_encoded, y_pred, target_names=le.classes_)
    print(f"classification Report for {name}:\n{report}\n")
    f1_score = classification_report(y_test_encoded, y_pred, output_dict=True)['weighted avg']['f1-score']
    f1_results.append((name, f1_score))

Training LogisticRegression...
classification Report for LogisticRegression:
                         precision    recall  f1-score   support

             functional       0.79      0.66      0.72      4144
functional_needs_repair       0.18      0.61      0.27       478
         non_functional       0.74      0.61      0.67      2731

               accuracy                           0.64      7353
              macro avg       0.57      0.63      0.55      7353
           weighted avg       0.73      0.64      0.67      7353


Training Decision Tree...
classification Report for Decision Tree:
                         precision    recall  f1-score   support

             functional       0.80      0.80      0.80      4144
functional_needs_repair       0.37      0.37      0.37       478
         non_functional       0.75      0.75      0.75      2731

               accuracy                           0.75      7353
              macro avg       0.64      0.64      0.64      7353
     

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:34:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


classification Report for XGBoost:
                         precision    recall  f1-score   support

             functional       0.85      0.73      0.78      4144
functional_needs_repair       0.27      0.69      0.39       478
         non_functional       0.79      0.74      0.77      2731

               accuracy                           0.73      7353
              macro avg       0.63      0.72      0.64      7353
           weighted avg       0.79      0.73      0.75      7353


Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003476 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 773
[LightGBM] [Info] Number of data points in the train set: 29411, number of used features: 71
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start t

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Note: class_weight='balanced' it helps in adjusting the algorithms loss functions to penalize misclassifications
of the minority class more heavily, giving underrepresented classes higher importance

## Evaluation Metrics for Imbalance Classificaton problems
### 1. Accuracy
Measure the overall correctness of a model.

Accuracy = Correct Predictions/Total Predictions

_Why accuracy is often misleading_
Consider a binary classification problem:
|Actual Class|Count|
|------------|-----|
|Negative    |900  |
|Positive    |   10|

The model every observation as Negative.

__Confusion Matrix_

|                   |Predicted Negative|predicted Positive|
|-------------------|------------------|------------------|
|Actual Negative    |990               |0                 |
|Actual Positive    |10                |0                 |

_Interpretation_
|                   |Predicted Negative|predicted Positive|
|-------------------|------------------|------------------|
|Actual Negative    |TN                |FP                 |
|Actual Positive    |FN                |TP                 |

Accuracy = (TP + TN)/(TP + FP + TN + FN)
Accuracy = (0 + 990)/(0 +0  + 990 + 10) > 990/100 = 99%

Despite achieving 99% accuracy, the model failed to identify a single positive case.
This is why accuracy should rarely be the primary metric for imbalanced  classification datasets.

Disadvantages; Misleading  for imbalanced datasets
Advantages; Easy to understand and uselful when classes  are balanced(roughly balanced)

2. Precision
- Measures  how many predicted positives were actually positive.

_Formula_
Presition = TP/(TP + FP)

__Question answered by precision_
of all the cases predicted as positive, how many were truly positive?

High precision means we have few false positives

Important when:
- Span filtering(Spam and Not Spam)
- Fraud Alerts
- Medical alerts where false alarms are expensive

3. Recall(Sensitivity/True Positive Rate)
Measures how many actual positives were found.
_Formula_

Recall = TP/(TP + FN)

*Recall answers the question*
Of all actual positives, how many did we detect?

_Example_
Actual fraud cases: 100

Detected: 80
False Negative: 20

Recall = 80(TP)/(80(TP) + 20(FN)) > 80/100 > 0.8

High Recall means 

Import when:
- Disease detection
- Fraud detection
- Safety systems

Missing a positive case is costly

##### Precision vs Recall
|Metric|Focus|
|------|-----|
||Precision|Reducing false positives|
|Recall|Reducing false negatives|

_Example:_ Cancer Screening
High Precision > Positive predictions are usually correct.
High Recall > Almost all cancer patients are identified.

4. F1-Score
Balances Precision and Recall.

_Formula_

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Why F1 score/ Harmonic mean**
If one metric is low, F1 becomes low.

Example: 
|Precision|Recall|F1|
|---------|------|--|
|0.90|0.90|0.90|
|0.90|0.10|0.18|

F1 penalizes imbalance between precision and recall.

we use F1 score when both False Negatives and False positives matter.

Example
- Fraud Detection
- Churn Predition
- Defect Detection
- Disease Classification

5. F2-Score
Generalization of f1.
When  Recall is twice as important as precision.
Useful in: 
- Disease detection
- Fraud detection
- Security threat detection

6. Macro-F1 Score/Macro Average
Computes the metric independency for each class and the average.

Examples:
|Classes|F1|
|-------|--|
|functional|0.95|
|non-functional|0.70|
|functional needs reapir|0.50|

Macro_f1 = (0.95 + 0.70 + 0.50)/3 = 0.72

Every class has equal importance.
_why Macro F1  is useful_
- Treats all classes equally
- Penalizes poor performance on minority classes
- Combines precision and recall
- Much more informative  than accuracy for imbalanced multiclass problems

7. Weighted Average  
Weights each class by its size.
|Class|Samples|
|-----|-------|
|A|5000|
|B|500|
|C|50|

Large classes influence the final score more.

**Task** Read on ROC-AUC curves, PR-AUC curve

#### Recommended metrics by problem type
|Problem Type|Best Metric|
|------------|-----------|
|Balanced Classification|Accuracy, ROC-AUC|
|Binary Imabalanced classification|F1, PR-AUC, Recall|
|Fraud Detection|Recall, F2, PR-AUC|
|MediacaL Diagnosis|Recall, F2|
|Spam Detection|Precision, F1|
|Customer churn|F1, PR-AUC|
|Muticlass Imbalance classification|Macro F1|
|Extreme Class Imbalance|PR-AC|


- Research on Hyper-parameter tuning
1. RandomSearchCV
2. GridSearchCV
3. Bayesian Optimization(optuna)

### Hyper parameter tuning.
Hyper parameters are settings that control how a machine learning algorithm learns from the data.

_Examples_
|Model|Hyperparameter|
|-----|--------------|
|||
||


##### Example Model
```Python
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

# parameter we want to tune
param_grid = {
    'n_estimators': [100, 200, 300],  # These are the number of trees
    'max_depth': [5, 10, 15],         # These are the numbers of internal nodes present
    'min_samples_split': [2, 5, 10]   # This is the minimum number of samples a node can have before moving to the next node
}
```

#### 1. GridSearch CV

Analogy:
Imagine searching for a treasure, gridsearch digs everywhere.
```Python
param_grid = {
    'n_estimators': [100, 200, 300],  # These are the number of trees
    'max_depth': [5, 10, 15],         # These are the numbers of internal nodes present
    'min_samples_split': [2, 5, 10]   # This is the minimum number of samples a node can have before moving to the next node
}
```

possible combination:
|n_estimators|max_depth|min_samples_split|
|------------|---------|-----------------|
|100|5|2|
|200|10|5|
|300|15|10|

Total combination:
3 * 3 * 3 = 27

In gridsearch every point is tested.

Advantages; Exhaustive search, Guaranteed to test every possible combination, Easy to understand  
Disadvantages: Computationality expensive, Becomes slow as parameters increase

#### RandomizerSearchCV 
Instead of testing every combination, RandomSearch a subset of combinations.

_Analogy_
Using the treasure hunt analogy, random search digs random location.

__Example_
```Python
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [5, 10, 15, 20, 25],
    'min_samples_split': [2, 5, 10, 20]
}
```
If we're to use gridsearch, we'd have the following total possible combinations:
5 * 5 * 4 = 100

but for RandomizedSearchCV May test only 20 of the 100 possible combinations.

_Advantages:_ Much faster, Works well with larger search spaces, usually finds near optimal solutions
_Disadvantages_ No Guarantee of finding the absolute best combination.

__GridSearch vs RandozedSearxh__


#### 3. Bayesian Optimization(Optuna)
It is based on Bayes 'Theorem which is a mathematical formula used to calculate
conditional probability

- bayesian optimization learns from previous experiments and intelligently chooses the next hyperparameters.

_Analogy_
Assume we're hunting for a treasure
Bayesian optimization follows the steps below:  
Dig somewhere > Find clues > Dig near promising locations.

_Optuna workflow_
Defining Search Space > Runs Trial > Evaluates Model > Update Knowledge > Choose Better parameters  
Note: This is a repetitive process based on number of iteration.

When to use Optuna(Bayesian Optimization):
Use when:
1. Large search spaces.
2. Competition projects
3. Production ML projects.
4. Deep Learning tuning

## STEP 7: Hyperparameter Tuning with Optuna

### GridSearch

In [ ]:
param_grid = {
    'n_estimator': [100, 200, 300, 400, 500],
    'max_depth': [5, 10, 15, 20, 25],
    'min_samples_split': [2, 5, 10, 20]
}

grid_search = GridSearchCV(full_pipeline, n_jobs=-1, param_distribution=param_grid, cv=5, )

### RandomizedSeachCV

In [ ]:
random_search = RandomizedSearchCV(full_pipeline, n_jobs=-1, param_distributions=param_grid, cv=5, n_iter=20, random_state=42)

grid_search.fit(X_train, y_train)

In [ ]:
best_estimator2 = random_search.best_estimator_
print("Best Hyperparameters", random_search.best_params_)

Optuna/Bayesian Optimization

In [ ]:
def objective(trial):
    model = RandomForestClassifier(
        n=trial.suggest_int('n_estimators', 100, 500),
        max_depth = trial.suggest_int('max_depth', 5, 20),
        min_samples_split =  trial.suggest_int('min_samples_split', 2, 5)
    )

    full_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    score = cross_val_score(
        full_pipeline,
         X_train,
          y_train_encoded,
           cv=5,
            scoring='f1_macro'
    ).mean()

    return score

study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=50)

# Disclaimer : The model we are saving at this stage is for deployment purposes and it doesn't mean it is the best performing
- The best performing model is RandomForestClassifier but the saved model is above 50mb(208mb)

In [18]:
pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', xgb.XGBClassifier(random_state=42))])

# Train model
pipeline.fit(X_train, y_train_encoded)        

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [19]:
y_pred2 = pipeline.predict(X_test)
print("Classification Report: \n", classification_report(y_test_encoded, y_pred2))

Classification Report: 
               precision    recall  f1-score   support

           0       0.79      0.89      0.84      4144
           1       0.53      0.23      0.32       478
           2       0.82      0.73      0.77      2731

    accuracy                           0.79      7353
   macro avg       0.71      0.62      0.64      7353
weighted avg       0.78      0.79      0.78      7353



In [20]:
# Save the XGBoost model
import joblib

joblib.dump(pipeline, "../models/best_xgb_model.joblib")




['../models/best_xgb_model.joblib']

In [ ]:
import os
xgb_model_file_path = "../models/best_xgb_model.joblib"

file_size = os.path.getsize(xgb_model_file_path)
print(f"XGBoost model Size: {file_size} bytes")

XGBoost model Size: 1050895 bytes


In [25]:
cleaned_df.head()

,amount_tsh,gps_height,population,age,month_recorded,permit,waterpoint_type_group,source_class,quantity,quality_group,payment_type,management_group,extraction_type_class,region,basin,status_group
0,6000.0,1390,109,12.0,3,False,communal_standpipe,groundwater,enough,good,annually,user-group,gravity,Iringa,Lake_Nyasa,functional
1,0.0,1399,280,3.0,3,True,communal_standpipe,surface,insufficient,good,never_pay,user-group,gravity,Mara,Lake_Victoria,functional
2,25.0,686,250,4.0,2,True,communal_standpipe,surface,enough,good,per_bucket,user-group,gravity,Manyara,Pangani,functional
3,0.0,263,58,27.0,1,True,communal_standpipe,groundwater,dry,good,never_pay,user-group,submersible,Mtwara,Ruvuma/Southern_Coast,non_functional
4,20.0,0,1,2.0,3,True,communal_standpipe,unknown,enough,salty,per_bucket,user-group,submersible,Tanga,Pangani,functional
